# 03 - Churn Label and Feature Engineering

In this notebook, we convert the EDA findings into modelling-ready customer-level features.

The goal is to create a clean customer-level dataset that can be used for churn prediction, recovery potential scoring, value-at-risk analysis, and retention prioritization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/LoyaltyRadar")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "data" / "outputs"
FIGURE_DIR = BASE_DIR / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [4]:
# Loading processed datasets from previous notebooks

df_monthly = pd.read_csv(PROCESSED_DIR / "02_customer_month_base.csv")
customer_activity_profile = pd.read_csv(OUTPUT_DIR / "02_customer_activity_profile.csv")
customer_redemption_profile = pd.read_csv(OUTPUT_DIR / "02_customer_redemption_profile.csv")
recent_activity_profile = pd.read_csv(OUTPUT_DIR / "02_recent_activity_profile.csv")
travel_break_profile = pd.read_csv(OUTPUT_DIR / "02_travel_break_profile.csv")
seasonal_profile = pd.read_csv(OUTPUT_DIR / "02_seasonal_profile.csv")

df_monthly["activity_date"] = pd.to_datetime(df_monthly["activity_date"])

print("Monthly base shape:", df_monthly.shape)
print("Customer activity shape:", customer_activity_profile.shape)
print("Recent activity shape:", recent_activity_profile.shape)
print("Travel break shape:", travel_break_profile.shape)
print("Seasonal profile shape:", seasonal_profile.shape)

Monthly base shape: (389065, 31)
Customer activity shape: (16737, 20)
Recent activity shape: (16737, 19)
Travel break shape: (16737, 20)
Seasonal profile shape: (16737, 19)


In [5]:
# Creating base customer-level feature table

features = customer_activity_profile.copy()

print("Feature table shape:", features.shape)
display(features.head())

Feature table shape: (16737, 20)


,loyalty_number,months_present,total_flights,active_months,total_distance,total_points_accumulated,total_points_redeemed,inactive_months,active_month_rate,avg_flights_per_active_month,clv,loyalty_card,province,city,gender,education,salary,enrollment_type,formal_churn,salary_missing
0,100018,24,46,18,81190,81190.0,1513,6,0.750000,2.555556,7919.20,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,Standard,0,0
1,100102,24,51,17,68918,68918.0,1195,7,0.708333,3.000000,2887.74,Nova,Ontario,Toronto,Male,College,NaN,Standard,0,1
2,100140,24,47,17,72856,72856.0,593,7,0.708333,2.764706,2838.07,Nova,British Columbia,Dawson Creek,Female,College,NaN,Standard,0,1
3,100214,24,22,9,38236,38236.0,861,15,0.375000,2.444444,4170.57,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,Standard,0,0
4,100272,24,37,13,54997,54997.0,1007,11,0.541667,2.846154,6622.05,Star,Ontario,Toronto,Female,Bachelor,91163.0,Standard,0,0


In [6]:
# Checking available columns before feature engineering

print(features.columns.tolist())

['loyalty_number', 'months_present', 'total_flights', 'active_months', 'total_distance', 'total_points_accumulated', 'total_points_redeemed', 'inactive_months', 'active_month_rate', 'avg_flights_per_active_month', 'clv', 'loyalty_card', 'province', 'city', 'gender', 'education', 'salary', 'enrollment_type', 'formal_churn', 'salary_missing']


In [7]:
# Adding recent activity and inactivity features

recent_cols = [
    "loyalty_number",
    "flights_last_3m",
    "flights_last_6m",
    "points_accumulated_last_6m",
    "points_redeemed_last_6m",
    "inactive_last_3m",
    "inactive_last_6m"
]

features = features.merge(
    recent_activity_profile[recent_cols],
    on="loyalty_number",
    how="left"
)

print("Feature table shape after recent activity merge:", features.shape)
display(features.head())

Feature table shape after recent activity merge: (16737, 26)


,loyalty_number,months_present,total_flights,active_months,total_distance,total_points_accumulated,total_points_redeemed,inactive_months,active_month_rate,avg_flights_per_active_month,clv,loyalty_card,province,city,gender,education,salary,enrollment_type,formal_churn,salary_missing,flights_last_3m,flights_last_6m,points_accumulated_last_6m,points_redeemed_last_6m,inactive_last_3m,inactive_last_6m
0,100018,24,46,18,81190,81190.0,1513,6,0.750000,2.555556,7919.20,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,Standard,0,0,10,17,36377.0,385,0,0
1,100102,24,51,17,68918,68918.0,1195,7,0.708333,3.000000,2887.74,Nova,Ontario,Toronto,Male,College,NaN,Standard,0,1,11,17,18795.0,0,0,0
2,100140,24,47,17,72856,72856.0,593,7,0.708333,2.764706,2838.07,Nova,British Columbia,Dawson Creek,Female,College,NaN,Standard,0,1,3,11,20488.0,0,0,0
3,100214,24,22,9,38236,38236.0,861,15,0.375000,2.444444,4170.57,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,Standard,0,0,3,9,14321.0,0,0,0
4,100272,24,37,13,54997,54997.0,1007,11,0.541667,2.846154,6622.05,Star,Ontario,Toronto,Female,Bachelor,91163.0,Standard,0,0,4,9,16412.0,0,0,0


In [8]:
# Adding travel pattern break features

travel_cols = [
    "loyalty_number",
    "flights_first_half_2018",
    "flights_second_half_2018",
    "flight_drop",
    "flight_drop_pct",
    "sharp_drop_flag"
]

features = features.merge(
    travel_break_profile[travel_cols],
    on="loyalty_number",
    how="left"
)

print("Feature table shape after travel break merge:", features.shape)
display(features.head())

Feature table shape after travel break merge: (16737, 31)


,loyalty_number,months_present,total_flights,active_months,total_distance,total_points_accumulated,total_points_redeemed,inactive_months,active_month_rate,avg_flights_per_active_month,clv,loyalty_card,province,city,gender,education,salary,enrollment_type,formal_churn,salary_missing,flights_last_3m,flights_last_6m,points_accumulated_last_6m,points_redeemed_last_6m,inactive_last_3m,inactive_last_6m,flights_first_half_2018,flights_second_half_2018,flight_drop,flight_drop_pct,sharp_drop_flag
0,100018,24,46,18,81190,81190.0,1513,6,0.750000,2.555556,7919.20,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,Standard,0,0,10,17,36377.0,385,0,0,5.0,17.0,-12.0,-2.400000,0
1,100102,24,51,17,68918,68918.0,1195,7,0.708333,3.000000,2887.74,Nova,Ontario,Toronto,Male,College,NaN,Standard,0,1,11,17,18795.0,0,0,0,9.0,17.0,-8.0,-0.888889,0
2,100140,24,47,17,72856,72856.0,593,7,0.708333,2.764706,2838.07,Nova,British Columbia,Dawson Creek,Female,College,NaN,Standard,0,1,3,11,20488.0,0,0,0,14.0,11.0,3.0,0.214286,0
3,100214,24,22,9,38236,38236.0,861,15,0.375000,2.444444,4170.57,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,Standard,0,0,3,9,14321.0,0,0,0,3.0,9.0,-6.0,-2.000000,0
4,100272,24,37,13,54997,54997.0,1007,11,0.541667,2.846154,6622.05,Star,Ontario,Toronto,Female,Bachelor,91163.0,Standard,0,0,4,9,16412.0,0,0,0,8.0,9.0,-1.0,-0.125000,0


In [9]:
# Adding seasonality features

seasonal_cols = [
    "loyalty_number",
    "q1_flights",
    "q2_flights",
    "q3_flights",
    "q4_flights",
    "seasonality_ratio",
    "seasonal_flag"
]

features = features.merge(
    seasonal_profile[seasonal_cols],
    on="loyalty_number",
    how="left"
)

print("Feature table shape after seasonality merge:", features.shape)
display(features.head())

Feature table shape after seasonality merge: (16737, 37)


,loyalty_number,months_present,total_flights,active_months,total_distance,total_points_accumulated,total_points_redeemed,inactive_months,active_month_rate,avg_flights_per_active_month,clv,loyalty_card,province,city,gender,education,salary,enrollment_type,formal_churn,salary_missing,flights_last_3m,flights_last_6m,points_accumulated_last_6m,points_redeemed_last_6m,inactive_last_3m,inactive_last_6m,flights_first_half_2018,flights_second_half_2018,flight_drop,flight_drop_pct,sharp_drop_flag,q1_flights,q2_flights,q3_flights,q4_flights,seasonality_ratio,seasonal_flag
0,100018,24,46,18,81190,81190.0,1513,6,0.750000,2.555556,7919.20,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,Standard,0,0,10,17,36377.0,385,0,0,5.0,17.0,-12.0,-2.400000,0,9,3,16,18,0.391304,0
1,100102,24,51,17,68918,68918.0,1195,7,0.708333,3.000000,2887.74,Nova,Ontario,Toronto,Male,College,NaN,Standard,0,1,11,17,18795.0,0,0,0,9.0,17.0,-8.0,-0.888889,0,10,10,13,18,0.352941,0
2,100140,24,47,17,72856,72856.0,593,7,0.708333,2.764706,2838.07,Nova,British Columbia,Dawson Creek,Female,College,NaN,Standard,0,1,3,11,20488.0,0,0,0,14.0,11.0,3.0,0.214286,0,9,13,15,10,0.319149,0
3,100214,24,22,9,38236,38236.0,861,15,0.375000,2.444444,4170.57,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,Standard,0,0,3,9,14321.0,0,0,0,3.0,9.0,-6.0,-2.000000,0,3,2,8,9,0.409091,0
4,100272,24,37,13,54997,54997.0,1007,11,0.541667,2.846154,6622.05,Star,Ontario,Toronto,Female,Bachelor,91163.0,Standard,0,0,4,9,16412.0,0,0,0,8.0,9.0,-1.0,-0.125000,0,7,15,7,8,0.405405,0


In [10]:
# Adding redemption behavior features

redemption_cols = [
    "loyalty_number",
    "redemption_months",
    "has_redeemed",
    "redemption_rate",
    "points_collector_flag"
]

features = features.merge(
    customer_redemption_profile[redemption_cols],
    on="loyalty_number",
    how="left"
)

print("Final merged feature table shape:", features.shape)
display(features.head())

Final merged feature table shape: (16737, 41)


,loyalty_number,months_present,total_flights,active_months,total_distance,total_points_accumulated,total_points_redeemed,inactive_months,active_month_rate,avg_flights_per_active_month,clv,loyalty_card,province,city,gender,education,salary,enrollment_type,formal_churn,salary_missing,flights_last_3m,flights_last_6m,points_accumulated_last_6m,points_redeemed_last_6m,inactive_last_3m,inactive_last_6m,flights_first_half_2018,flights_second_half_2018,flight_drop,flight_drop_pct,sharp_drop_flag,q1_flights,q2_flights,q3_flights,q4_flights,seasonality_ratio,seasonal_flag,redemption_months,has_redeemed,redemption_rate,points_collector_flag
0,100018,24,46,18,81190,81190.0,1513,6,0.750000,2.555556,7919.20,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,Standard,0,0,10,17,36377.0,385,0,0,5.0,17.0,-12.0,-2.400000,0,9,3,16,18,0.391304,0,3,True,0.018635,1
1,100102,24,51,17,68918,68918.0,1195,7,0.708333,3.000000,2887.74,Nova,Ontario,Toronto,Male,College,NaN,Standard,0,1,11,17,18795.0,0,0,0,9.0,17.0,-8.0,-0.888889,0,10,10,13,18,0.352941,0,2,True,0.017339,1
2,100140,24,47,17,72856,72856.0,593,7,0.708333,2.764706,2838.07,Nova,British Columbia,Dawson Creek,Female,College,NaN,Standard,0,1,3,11,20488.0,0,0,0,14.0,11.0,3.0,0.214286,0,9,13,15,10,0.319149,0,1,True,0.008139,1
3,100214,24,22,9,38236,38236.0,861,15,0.375000,2.444444,4170.57,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,Standard,0,0,3,9,14321.0,0,0,0,3.0,9.0,-6.0,-2.000000,0,3,2,8,9,0.409091,0,2,True,0.022518,0
4,100272,24,37,13,54997,54997.0,1007,11,0.541667,2.846154,6622.05,Star,Ontario,Toronto,Female,Bachelor,91163.0,Standard,0,0,4,9,16412.0,0,0,0,8.0,9.0,-1.0,-0.125000,0,7,15,7,8,0.405405,0,2,True,0.018310,1


In [11]:
# Checking missing values after merging feature tables

missing_after_merge = features.isnull().sum().sort_values(ascending=False)
display(missing_after_merge[missing_after_merge > 0])

,0
salary,4238


In [12]:
# Handling salary missing values using median salary by education

features["salary_missing"] = features["salary"].isnull().astype(int)

education_salary_median = features.groupby("education")["salary"].median()

features["salary_imputed"] = features["salary"]

for edu in features["education"].unique():
    edu_median = education_salary_median.loc[edu]
    features.loc[
        (features["education"] == edu) & (features["salary_imputed"].isnull()),
        "salary_imputed"
    ] = edu_median

overall_salary_median = features["salary"].median()

features["salary_imputed"] = features["salary_imputed"].fillna(overall_salary_median)

print("Missing salary before:", features["salary"].isnull().sum())
print("Missing salary after imputation:", features["salary_imputed"].isnull().sum())

display(features[["education", "salary", "salary_missing", "salary_imputed"]].head(10))

Missing salary before: 4238
Missing salary after imputation: 0


,education,salary,salary_missing,salary_imputed
0,Bachelor,92552.0,0,92552.0
1,College,NaN,1,73455.0
2,College,NaN,1,73455.0
3,Bachelor,63253.0,0,63253.0
4,Bachelor,91163.0,0,91163.0
5,Bachelor,70323.0,0,70323.0
6,Bachelor,76849.0,0,76849.0
7,Bachelor,69695.0,0,69695.0
8,Bachelor,63478.0,0,63478.0
9,Bachelor,75638.0,0,75638.0


We do not drop salary-missing rows because that would remove all College customers.
So we keep salary_missing as a signal and create salary_imputed for modelling.

In [13]:
# Creating CLV value tiers

features["clv_tier"] = pd.qcut(
    features["clv"],
    q=4,
    labels=["Low Value", "Mid Value", "High Value", "Premium Value"]
)

features["high_value_flag"] = (
    features["clv_tier"].isin(["High Value", "Premium Value"])
).astype(int)

features["premium_value_flag"] = (
    features["clv_tier"] == "Premium Value"
).astype(int)

display(
    features["clv_tier"]
    .value_counts()
    .sort_index()
)

,count
clv_tier,
Low Value,4186
Mid Value,4192
High Value,4176
Premium Value,4183


In [14]:
# Checking CLV tier quality

clv_tier_summary = (
    features
    .groupby("clv_tier")
    .agg(
        customers=("loyalty_number", "count"),
        min_clv=("clv", "min"),
        avg_clv=("clv", "mean"),
        max_clv=("clv", "max"),
        formal_churn_rate=("formal_churn", "mean")
    )
    .round(3)
)

display(clv_tier_summary)

/tmp/ipykernel_5473/708539799.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("clv_tier")


,customers,min_clv,avg_clv,max_clv,formal_churn_rate
clv_tier,,,,,
Low Value,4186,1898.01,2905.173,3980.84,0.119
Mid Value,4192,3981.78,4949.019,5780.18,0.127
High Value,4176,5781.02,7433.166,8940.58,0.124
Premium Value,4183,8945.69,16677.484,83325.38,0.124


Observation:
CLV tiers divide customers into balanced value groups. Premium Value customers have much higher average CLV than other tiers, but formal churn rates are similar across all CLV tiers. This confirms that CLV should be used for value prioritization rather than as a standalone churn-risk indicator. High-value customers are important when they become behaviorally inactive, because their value at risk is higher.

In [15]:
# Creating behavioral churn label based on sustained recent inactivity
# behavioral_churn = 1 if customer had no flights in the last 6 months
features["behavioral_churn"] = (
    features["inactive_last_6m"] == 1
).astype(int)

behavioral_churn_summary = (
    features
    .groupby("behavioral_churn")
    .agg(
        customers=("loyalty_number", "count"),
        avg_clv=("clv", "mean"),
        formal_churn_rate=("formal_churn", "mean"),
        avg_active_month_rate=("active_month_rate", "mean"),
        avg_total_flights=("total_flights", "mean")
    )
    .round(3)
)

display(behavioral_churn_summary)

,customers,avg_clv,formal_churn_rate,avg_active_month_rate,avg_total_flights
behavioral_churn,,,,,
0,14268,7965.216,0.023,0.525,34.77
1,2469,8125.742,0.706,0.080,5.15


In [16]:
# Comparing formal churn and behavioral churn

churn_comparison = pd.crosstab(
    features["formal_churn"],
    features["behavioral_churn"],
    margins=True
)

display(churn_comparison)

churn_comparison_pct = pd.crosstab(
    features["formal_churn"],
    features["behavioral_churn"],
    normalize="index"
).round(3) * 100

display(churn_comparison_pct)

behavioral_churn,0,1,All
formal_churn,,,
0,13943,727,14670
1,325,1742,2067
All,14268,2469,16737


behavioral_churn,0,1
formal_churn,,
0,95.0,5.0
1,15.7,84.3


Observation:
The behavioral churn label based on 6-month recent inactivity aligns strongly with formal churn. Around 84.3% of formally churned customers are also behaviorally churned, while 95.0% of formally non-churned customers are behaviorally active. This confirms that sustained recent inactivity is a meaningful proxy for churn behavior. However, the mismatch groups are also important: some formally active customers already look behaviorally churned, and some formally churned customers still show recent activity. This supports building a risk-based retention system instead of relying only on formal cancellation status.

In [17]:
# Creating churn status groups using formal and behavioral churn

def create_churn_status(row):
    if row["formal_churn"] == 1 and row["behavioral_churn"] == 1:
        return "Confirmed Churn"
    elif row["formal_churn"] == 0 and row["behavioral_churn"] == 1:
        return "Silent Risk"
    elif row["formal_churn"] == 1 and row["behavioral_churn"] == 0:
        return "Recently Active Churn"
    else:
        return "Healthy Active"

features["churn_status_group"] = features.apply(create_churn_status, axis=1)

churn_status_summary = (
    features
    .groupby("churn_status_group")
    .agg(
        customers=("loyalty_number", "count"),
        avg_clv=("clv", "mean"),
        total_clv=("clv", "sum"),
        avg_active_month_rate=("active_month_rate", "mean"),
        avg_total_flights=("total_flights", "mean")
    )
    .round(3)
    .sort_values("customers", ascending=False)
)

display(churn_status_summary)

,customers,avg_clv,total_clv,avg_active_month_rate,avg_total_flights
churn_status_group,,,,,
Healthy Active,13943,7967.119,1.110855e+08,0.527,34.837
Confirmed Churn,1742,8178.083,1.424622e+07,0.094,6.072
Silent Risk,727,8000.327,5.816238e+06,0.046,2.939
Recently Active Churn,325,7883.578,2.562163e+06,0.414,31.858


Observation:
Combining formal churn and behavioral churn creates more useful customer states than using cancellation status alone. The Silent Risk group contains 727 customers who have not formally churned but show sustained recent inactivity. This group represents around 5.82 million CLV and has very low historical engagement, making it a strong candidate for proactive retention campaigns. This supports the business value of identifying behavioral risk before formal cancellation occurs.

In [18]:
# Creating Loyalty Value at Risk metric

features["loyalty_value_at_risk"] = np.where(
    features["behavioral_churn"] == 1,
    features["clv"],
    0
)

value_at_risk_summary = (
    features
    .groupby("churn_status_group")
    .agg(
        customers=("loyalty_number", "count"),
        total_clv=("clv", "sum"),
        total_value_at_risk=("loyalty_value_at_risk", "sum"),
        avg_value_at_risk=("loyalty_value_at_risk", "mean")
    )
    .round(2)
    .sort_values("total_value_at_risk", ascending=False)
)

display(value_at_risk_summary)

,customers,total_clv,total_value_at_risk,avg_value_at_risk
churn_status_group,,,,
Confirmed Churn,1742,1.424622e+07,14246219.88,8178.08
Silent Risk,727,5.816238e+06,5816237.99,8000.33
Healthy Active,13943,1.110855e+08,0.00,0.00
Recently Active Churn,325,2.562163e+06,0.00,0.00


In [19]:
# Calculating value-at-risk share

total_clv = features["clv"].sum()
total_value_at_risk = features["loyalty_value_at_risk"].sum()

print("Total CLV:", round(total_clv, 2))
print("Total Loyalty Value at Risk:", round(total_value_at_risk, 2))
print("Value at Risk Share (%):", round(total_value_at_risk / total_clv * 100, 2))

Total CLV: 133710161.32
Total Loyalty Value at Risk: 20062457.87
Value at Risk Share (%): 15.0


Observation:
The total observed CLV in the customer base is around 133.71 million. Customers flagged as behaviorally churned account for around 20.06 million CLV, which represents 15.0% of total customer value. This shows that recent inactivity is not only a churn signal but also a meaningful business risk. Loyalty Value at Risk can therefore be used as a dashboard KPI for retention prioritization.

In [20]:
# Creating a simple retention priority score

features["retention_priority_score"] = (
    features["behavioral_churn"] * 40 +
    features["premium_value_flag"] * 20 +
    features["sharp_drop_flag"] * 15 +
    features["seasonal_flag"] * 5 +
    features["points_collector_flag"] * 10 +
    features["inactive_last_3m"] * 10
)

features["retention_priority_score"] = features["retention_priority_score"].clip(0, 100)

display(
    features[
        ["loyalty_number", "clv", "clv_tier", "behavioral_churn",
         "sharp_drop_flag", "seasonal_flag", "points_collector_flag",
         "retention_priority_score"]
    ]
    .sort_values("retention_priority_score", ascending=False)
    .head(20)
)

,loyalty_number,clv,clv_tier,behavioral_churn,sharp_drop_flag,seasonal_flag,points_collector_flag,retention_priority_score
223,112142,16272.74,Premium Value,1,1,0,1,95
7184,486708,23295.06,Premium Value,1,1,0,1,95
3832,307047,13211.01,Premium Value,1,1,0,1,95
574,131253,19327.87,Premium Value,1,1,0,1,95
15026,907077,10208.93,Premium Value,1,1,0,1,95
8267,545410,13918.53,Premium Value,1,1,0,1,95
10627,670836,19079.10,Premium Value,1,1,1,0,90
16638,994679,13055.57,Premium Value,1,1,1,0,90
2715,247391,12734.21,Premium Value,1,1,1,0,90
340,118376,20455.25,Premium Value,1,1,1,0,90


In [21]:
# Summarizing retention priority score

priority_summary = (
    features["retention_priority_score"]
    .describe()
    .round(2)
)

display(priority_summary)

,retention_priority_score
count,16737.00
mean,17.55
std,19.60
min,0.00
25%,0.00
50%,10.00
75%,30.00
max,95.00


Observation:
The rule-based retention priority score creates a practical ranking of customers. Most customers have low scores, while a smaller group receives high scores based on behavioral churn, premium value, travel pattern break, seasonality, points collector behavior, and recent inactivity. This score is not the final ML model, but it provides a business-friendly baseline for prioritizing retention actions.

In [22]:
# Creating retention priority bands

def create_priority_band(score):
    if score >= 70:
        return "Critical Priority"
    elif score >= 45:
        return "High Priority"
    elif score >= 20:
        return "Medium Priority"
    else:
        return "Low Priority"

features["priority_band"] = features["retention_priority_score"].apply(create_priority_band)

priority_band_summary = (
    features
    .groupby("priority_band")
    .agg(
        customers=("loyalty_number", "count"),
        avg_score=("retention_priority_score", "mean"),
        avg_clv=("clv", "mean"),
        total_value_at_risk=("loyalty_value_at_risk", "sum"),
        formal_churn_rate=("formal_churn", "mean"),
        behavioral_churn_rate=("behavioral_churn", "mean")
    )
    .round(3)
    .sort_values("avg_score", ascending=False)
)

display(priority_band_summary)

,customers,avg_score,avg_clv,total_value_at_risk,formal_churn_rate,behavioral_churn_rate
priority_band,,,,,,
Critical Priority,657,72.154,16447.272,10805857.70,0.721,1.000
High Priority,1880,51.628,5501.185,9256600.17,0.681,0.964
Medium Priority,3886,25.151,15456.798,0.00,0.030,0.000
Low Priority,10314,4.989,5089.874,0.00,0.019,0.000


Observation:
The priority bands create a clear retention targeting structure. Critical Priority customers are a small group of 657 customers, but they have very high average CLV, 100% behavioral churn rate, and around 10.81 million value at risk. High Priority customers add another 1,880 customers with high churn risk and around 9.26 million value at risk. This shows that a focused retention strategy can target a relatively small customer group while covering the full observed Loyalty Value at Risk.

In [23]:
# Creating final modelling target

features["target_churn"] = features["behavioral_churn"]

target_summary = (
    features["target_churn"]
    .value_counts()
    .to_frame("customers")
)

target_summary["percentage"] = (
    target_summary["customers"] / target_summary["customers"].sum() * 100
).round(2)

display(target_summary)

,customers,percentage
target_churn,,
0,14268,85.25
1,2469,14.75


Observation:
The final modelling target is behavioral churn, defined as no flights in the last 6 months. This target flags 2,469 customers, or around 14.75% of the customer base. It is more useful for proactive retention than formal churn alone because it identifies customers showing risky inactivity before or alongside cancellation.

In [24]:
# Selecting final columns for modelling and business scoring

final_columns = [
    "loyalty_number",

    # Target labels
    "target_churn",
    "behavioral_churn",
    "formal_churn",
    "churn_status_group",

    # Activity features
    "months_present",
    "total_flights",
    "active_months",
    "inactive_months",
    "active_month_rate",
    "avg_flights_per_active_month",
    "total_distance",

    # Recent activity features
    "flights_last_3m",
    "flights_last_6m",
    "points_accumulated_last_6m",
    "points_redeemed_last_6m",
    "inactive_last_3m",
    "inactive_last_6m",

    # Travel break features
    "flights_first_half_2018",
    "flights_second_half_2018",
    "flight_drop",
    "flight_drop_pct",
    "sharp_drop_flag",

    # Seasonality features
    "q1_flights",
    "q2_flights",
    "q3_flights",
    "q4_flights",
    "seasonality_ratio",
    "seasonal_flag",

    # Points behavior
    "total_points_accumulated",
    "total_points_redeemed",
    "redemption_months",
    "has_redeemed",
    "redemption_rate",
    "points_collector_flag",

    # Value features
    "clv",
    "clv_tier",
    "high_value_flag",
    "premium_value_flag",
    "loyalty_value_at_risk",
    "retention_priority_score",
    "priority_band",

    # Profile features
    "loyalty_card",
    "province",
    "city",
    "gender",
    "education",
    "salary_imputed",
    "salary_missing",
    "enrollment_type"
]

final_features = features[final_columns].copy()

print("Final modelling dataset shape:", final_features.shape)
display(final_features.head())

Final modelling dataset shape: (16737, 50)


,loyalty_number,target_churn,behavioral_churn,formal_churn,churn_status_group,months_present,total_flights,active_months,inactive_months,active_month_rate,avg_flights_per_active_month,total_distance,flights_last_3m,flights_last_6m,points_accumulated_last_6m,points_redeemed_last_6m,inactive_last_3m,inactive_last_6m,flights_first_half_2018,flights_second_half_2018,flight_drop,flight_drop_pct,sharp_drop_flag,q1_flights,q2_flights,q3_flights,q4_flights,seasonality_ratio,seasonal_flag,total_points_accumulated,total_points_redeemed,redemption_months,has_redeemed,redemption_rate,points_collector_flag,clv,clv_tier,high_value_flag,premium_value_flag,loyalty_value_at_risk,retention_priority_score,priority_band,loyalty_card,province,city,gender,education,salary_imputed,salary_missing,enrollment_type
0,100018,0,0,0,Healthy Active,24,46,18,6,0.750000,2.555556,81190,10,17,36377.0,385,0,0,5.0,17.0,-12.0,-2.400000,0,9,3,16,18,0.391304,0,81190.0,1513,3,True,0.018635,1,7919.20,High Value,1,0,0.0,10,Low Priority,Aurora,Alberta,Edmonton,Female,Bachelor,92552.0,0,Standard
1,100102,0,0,0,Healthy Active,24,51,17,7,0.708333,3.000000,68918,11,17,18795.0,0,0,0,9.0,17.0,-8.0,-0.888889,0,10,10,13,18,0.352941,0,68918.0,1195,2,True,0.017339,1,2887.74,Low Value,0,0,0.0,10,Low Priority,Nova,Ontario,Toronto,Male,College,73455.0,1,Standard
2,100140,0,0,0,Healthy Active,24,47,17,7,0.708333,2.764706,72856,3,11,20488.0,0,0,0,14.0,11.0,3.0,0.214286,0,9,13,15,10,0.319149,0,72856.0,593,1,True,0.008139,1,2838.07,Low Value,0,0,0.0,10,Low Priority,Nova,British Columbia,Dawson Creek,Female,College,73455.0,1,Standard
3,100214,0,0,0,Healthy Active,24,22,9,15,0.375000,2.444444,38236,3,9,14321.0,0,0,0,3.0,9.0,-6.0,-2.000000,0,3,2,8,9,0.409091,0,38236.0,861,2,True,0.022518,0,4170.57,Mid Value,0,0,0.0,0,Low Priority,Star,British Columbia,Vancouver,Male,Bachelor,63253.0,0,Standard
4,100272,0,0,0,Healthy Active,24,37,13,11,0.541667,2.846154,54997,4,9,16412.0,0,0,0,8.0,9.0,-1.0,-0.125000,0,7,15,7,8,0.405405,0,54997.0,1007,2,True,0.018310,1,6622.05,High Value,1,0,0.0,10,Low Priority,Star,Ontario,Toronto,Female,Bachelor,91163.0,0,Standard


In [25]:
# Checking final dataset quality

print("Shape:", final_features.shape)
print("Unique customers:", final_features["loyalty_number"].nunique())
print("Duplicate loyalty numbers:", final_features["loyalty_number"].duplicated().sum())

print("\nMissing values:")
display(final_features.isnull().sum()[final_features.isnull().sum() > 0])

print("\nTarget distribution:")
display(final_features["target_churn"].value_counts(normalize=True).round(3) * 100)

Shape: (16737, 50)
Unique customers: 16737
Duplicate loyalty numbers: 0

Missing values:


,0



Target distribution:


,proportion
target_churn,
0,85.2
1,14.8


Observation:
The final modelling dataset contains 16,737 unique customers and 50 columns covering churn labels, activity behavior, recent inactivity, travel pattern break, seasonality, points behavior, CLV value, priority scoring, and customer profile features. There are no duplicate customers and no missing values after feature engineering. The final target churn rate is 14.8%, which is moderately imbalanced but still suitable for classification modelling with proper evaluation metrics such as precision, recall, F1-score, ROC-AUC, PR-AUC, and lift.

In [26]:
# Saving final modelling and scoring datasets

final_features.to_csv(
    PROCESSED_DIR / "03_final_customer_features.csv",
    index=False
)

priority_band_summary.to_csv(
    OUTPUT_DIR / "03_priority_band_summary.csv"
)

churn_status_summary.to_csv(
    OUTPUT_DIR / "03_churn_status_summary.csv"
)

value_at_risk_summary.to_csv(
    OUTPUT_DIR / "03_value_at_risk_summary.csv"
)

clv_tier_summary.to_csv(
    OUTPUT_DIR / "03_clv_tier_summary.csv"
)

print("Saved NB03 outputs successfully.")
print("Final customer features saved at:", PROCESSED_DIR / "03_final_customer_features.csv")

Saved NB03 outputs successfully.
Final customer features saved at: /content/drive/MyDrive/LoyaltyRadar/data/processed/03_final_customer_features.csv


In [27]:
# Summarizing Notebook 03 results

print("NB03 SUMMARY")
print("-" * 50)

print("Final customer feature dataset shape:", final_features.shape)
print("Unique customers:", final_features["loyalty_number"].nunique())

print("\nTarget churn distribution:")
print(final_features["target_churn"].value_counts(normalize=True).round(3) * 100)

print("\nKey business metrics:")
print("Total CLV:", round(total_clv, 2))
print("Total Loyalty Value at Risk:", round(total_value_at_risk, 2))
print("Value at Risk Share (%):", round(total_value_at_risk / total_clv * 100, 2))

print("\nPriority band distribution:")
print(final_features["priority_band"].value_counts())

print("\nNotebook 03 completed successfully.")

NB03 SUMMARY
--------------------------------------------------
Final customer feature dataset shape: (16737, 50)
Unique customers: 16737

Target churn distribution:
target_churn
0    85.2
1    14.8
Name: proportion, dtype: float64

Key business metrics:
Total CLV: 133710161.32
Total Loyalty Value at Risk: 20062457.87
Value at Risk Share (%): 15.0

Priority band distribution:
priority_band
Low Priority         10314
Medium Priority       3886
High Priority         1880
Critical Priority      657
Name: count, dtype: int64

Notebook 03 completed successfully.
